# Etapa 8. DSSAT CERES-Maize e ky emergente

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/08_dssat_ky_emergente.ipynb)

O **DSSAT CERES-Maize** é um modelo de processo: simula fotossíntese, crescimento de folhas, raízes e grãos, dia a
dia. Aqui ele é usado como uma segunda opinião, em poucos municípios:

1. roda o milho **irrigado** (sem falta de água) e em **sequeiro**;
2. a razão sequeiro/irrigado é a perda pelo déficit hídrico segundo o DSSAT;
3. ajusta-se o ky que faz o ISNA do MZA-FAO reproduzir essa perda: é o **ky emergente**.

Se o ky emergente for parecido com o ky calibrado (Etapa 6), o resultado ganha credibilidade. Se não, a explicação
da diferença é um resultado em si.

> No Colab, o DSSAT roda direto: o pacote `DSSATTools` já traz o programa para Linux.

In [ ]:
# ==========================================================================================
# CÉLULA 0. PREPARAR O AMBIENTE (rode sempre primeiro, toda vez que abrir o notebook)
# ==========================================================================================
# No Colab, esta célula:
#   1) instala as bibliotecas que o Colab não tem;
#   2) copia o projeto do GitHub (dados + funções) para a máquina do Colab;
#   3) conecta o seu Google Drive, onde os resultados ficam guardados entre uma sessão e outra.
# Fora do Colab (no computador do orientador), ela só acerta as pastas.
import os
import sys
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules        # True quando o notebook roda no Colab

if NO_COLAB:
    !pip -q install agrometeorologiapy==1.0.1 DSSATTools==3.0.2
    if Path("/content/milho_oeste").exists():
        !git -C /content/milho_oeste pull -q     # já copiado nesta sessão: só atualiza
    else:
        !git clone -q https://github.com/fcoliveira-utfpr/milho_oeste.git /content/milho_oeste
    os.chdir("/content/milho_oeste")             # entra na pasta do projeto
    from google.colab import drive
    drive.mount("/content/drive")                # pede autorização na primeira vez
    SAIDA = Path("/content/drive/MyDrive/milho_oeste/resultados")
else:
    if Path.cwd().name == "notebooks":
        os.chdir("..")
    SAIDA = Path("resultados")

SAIDA.mkdir(parents=True, exist_ok=True)         # cria a pasta de resultados, se não existir
sys.path.append("src")                           # permite importar as funções do projeto (pasta src/)
print("Pasta do projeto:", Path.cwd())
print("Resultados em   :", SAIDA)

In [ ]:
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize

warnings.filterwarnings("ignore", category=DeprecationWarning)   # aviso de versão do DSSATTools
import dssat_oeste as D
import mza_oeste as M

clima = pd.read_parquet("dados/clima_oeste.parquet")
mun = pd.read_csv("dados/municipios_oeste.csv")
camadas = pd.read_csv("dados/solo_camadas_oeste.csv")
sim = pd.read_parquet(SAIDA / "simulacao_mza_oeste.parquet")

## 1. Os municípios

Quatro municípios conhecidos e o de solo **mais arenoso** do Oeste (menor CAD), para cobrir a variação de solo.

In [ ]:
nomes = ["Santa Helena", "Cascavel", "Toledo", "Guaíra"]
arenoso = mun.sort_values("areia_pct", ascending=False).iloc[0]["municipio"]
escolhidos = mun[mun["municipio"].isin(nomes + [arenoso])]
escolhidos[["municipio", "altitude_m", "dta_mm_m", "areia_pct", "argila_pct"]]

## 2. Calibrar a fenologia da cultivar

A cultivar genérica do DSSAT floresce cedo demais para um híbrido de safrinha. Os **coeficientes genéticos**
controlam isso:

| Coeficiente | O que controla |
|---|---|
| P1 | soma térmica da emergência ao fim da fase juvenil (°C dia): **atrasa o florescimento** |
| P5 | soma térmica do florescimento à maturação (°C dia): **alonga o enchimento** |
| G2 | número máximo de grãos por planta |
| G3 | taxa de enchimento dos grãos (mg/dia) |

**Alvo** para um híbrido precoce semeado em fevereiro (conferir no ZARC e nos catálogos dos híbridos mais semeados
no Oeste): florescimento por volta de **60 dias** e maturação fisiológica por volta de **120 dias**.

Rode a célula, compare com o alvo e ajuste os números. Cada rodada leva poucos segundos.

In [ ]:
COEF = {"P1": 280, "P5": 900, "G2": 750, "G3": 8.5}      # <<< ajuste aqui

sh = mun[mun["municipio"] == "Santa Helena"].iloc[0]
clima_sh = clima[clima["cod_ibge"] == sh["cod_ibge"]].set_index("data")
teste = D.simular(clima_sh, camadas[camadas["cod_ibge"] == sh["cod_ibge"]], sh, [(2, 5)],
                  irrigado=True, coeficientes=COEF, n_anos=10)
teste[["dias_antese", "dias_maturacao", "rend"]].describe().loc[["mean", "min", "max"]].round(0)

**Produtividade:** a do irrigado deve ficar perto do topo dos ensaios de cultivares do IDR-Paraná (Etapa 5). Se
ficar muito acima ou abaixo, ajuste G2 e G3. Anote os coeficientes finais e a justificativa: isso vai para o
Material e Métodos.

## 3. Sequeiro e irrigado nos municípios escolhidos

As 4 datas da janela × 27 anos × 2 condições × 5 municípios. Leva alguns minutos.

In [ ]:
datas_janela = [(int(d[3:]), int(d[:2])) for d in M.JANELA]       # "15/01" -> (1, 15)
tabelas = []
for m in escolhidos.itertuples():
    m = mun.loc[m.Index]                                           # a linha completa do município
    clima_m = clima[clima["cod_ibge"] == m["cod_ibge"]].set_index("data")
    cam_m = camadas[camadas["cod_ibge"] == m["cod_ibge"]]
    for irrigado in [False, True]:
        t = D.simular(clima_m, cam_m, m, datas_janela, irrigado=irrigado, coeficientes=COEF)
        tabelas.append(t.assign(condicao="irrigado" if irrigado else "sequeiro", municipio=m["municipio"]))
    print("ok:", m["municipio"])
dssat = pd.concat(tabelas, ignore_index=True)
dssat.to_parquet(SAIDA / "dssat_oeste.parquet", index=False)

In [ ]:
larga = dssat.pivot_table(index=["cod_ibge", "municipio", "ano", "semeadura"], columns="condicao",
                          values="rend").reset_index()
larga["rel_D"] = (larga["sequeiro"] / larga["irrigado"]).clip(upper=1.0)    # perda pelo déficit, segundo o DSSAT
larga.groupby("municipio")[["irrigado", "sequeiro", "rel_D"]].mean().round(2)

## 4. Ky emergente

Junta-se a perda do DSSAT (`rel_D`) com o ISNA de cada fase do MZA-FAO **no mesmo município, ano e data**, e
ajusta-se o ky que minimiza o erro entre `rel_D` e a eficiência climática do MZA (mesma função da Etapa 6).

In [ ]:
from avaliacao import avaliar, rmse

dados = larga.merge(sim, on=["cod_ibge", "ano", "semeadura"])
x_dk = [M.KY_DK["vegetativo"], M.KY_DK["floracao"], M.KY_DK["enchimento"]]

def ky_de(x):
    return {**M.KY_DK, "vegetativo": x[0], "floracao": x[1], "enchimento": x[2]}

def erro_D(x):
    return rmse(dados["rel_D"], M.ec(dados, ky_de(x)))

r = minimize(erro_D, x0=x_dk, bounds=[(0.05, 2.5)] * 3, method="L-BFGS-B")
KY_EMERGENTE = ky_de(r.x)
print(f"ky emergente: veg = {r.x[0]:.2f}, flor = {r.x[1]:.2f}, ench = {r.x[2]:.2f}   RMSE = {r.fun:.3f}")

**Como interpretar.** Se o ky emergente cair no limite inferior (0,05), o DSSAT está vendo **muito menos déficit**
do que o MZA-FAO: o perfil de solo do DSSAT tem 180 cm e a raiz explora água em profundidade, enquanto no MZA a raiz
chega a 0,40 m. Compare o `rel_D` médio (tabela acima) com a EC média do MZA no mesmo município. Esse contraste é
um resultado: ele mostra o quanto o nível do déficit depende da profundidade de raízes, e justifica (ou não) a
escolha do nível "normalizado" na Etapa 7. Um teste que vale fazer: em `src/mza_oeste.py`, mude `Z_MAX` para 0,7 ou
1,0 m, refaça a Etapa 4 e veja como mudam o ISNA e o ky emergente.

In [ ]:
print(f"Perda média pelo déficit segundo o DSSAT (1 − sequeiro/irrigado): {1 - dados['rel_D'].mean():.2f}")
print(f"Perda média segundo o MZA com o ky de D&K (1 − EC):              {1 - M.ec(dados, M.KY_DK).mean():.2f}")
print(f"ISNA médio do MZA na floração: {dados['ISNA_floracao'].mean():.2f};  no enchimento: {dados['ISNA_enchimento'].mean():.2f}")

In [ ]:
cal = json.load(open(SAIDA / "ky_calibrado.json"))
tres = pd.DataFrame({"Doorenbos & Kassam": M.KY_DK, "Calibrado (IBGE)": cal["ky"],
                     "Emergente (DSSAT)": KY_EMERGENTE}).round(2)
tres.to_csv(SAIDA / "tabela_tres_ky.csv")
tres

## 5. PP e PA dos dois modelos

In [ ]:
FATOR_PP = json.load(open(SAIDA / "parametros.json"))["FATOR_PP"]
dados["PP_mza"] = dados["PPf"] * FATOR_PP
dados["PA_mza"] = dados["PP_mza"] * M.ec(dados, cal["ky"])

fig, axs = plt.subplots(1, 2, figsize=(11, 5))
for ax, (x, y, t) in zip(axs, [("irrigado", "PP_mza", "Potencial: DSSAT irrigado × PP do MZA"),
                               ("sequeiro", "PA_mza", "Atingível: DSSAT sequeiro × PA do MZA")]):
    ax.scatter(dados[x], dados[y], s=10, alpha=0.5)
    lim = [0, max(dados[x].max(), dados[y].max())]
    ax.plot(lim, lim, "k--", lw=0.8)
    ax.set_xlabel(f"DSSAT {x} (kg/ha)"); ax.set_ylabel(f"MZA (kg/ha)"); ax.set_title(t, fontsize=10)
fig.savefig(SAIDA / "fig_dssat_vs_mza.png", dpi=300, bbox_inches="tight")
plt.show()
pd.DataFrame({"PP": avaliar(dados["irrigado"], dados["PP_mza"]),
              "PA": avaliar(dados["sequeiro"], dados["PA_mza"])}).T.round(2)

### O que escrever

No Material e Métodos, a subseção do **DSSAT CERES-Maize**: municípios, coeficientes genéticos e como foram
ajustados, manejo (população, adubação, irrigação automática) e o cálculo do ky emergente. Nos Resultados, a
tabela dos três ky e a comparação entre os modelos.